In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


# 1. 데이터 로딩
cust_df = pd.read_csv("../data/santander-customer-satisfaction/train.csv", encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)
 

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [3]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [4]:
# 2. 불균형 확인
print(cust_df['TARGET'].value_counts())
unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()
print('unsatisfied 비율은 {0:.2f}'.format((unsatisfied_cnt / total_cnt)))

TARGET
0    73012
1     3008
Name: count, dtype: int64
unsatisfied 비율은 0.04


In [5]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [6]:
# 3. 이상값 탐지(var3의 min: -999999)
cust_df.describe()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [7]:
# 4-1. 전처리(var3의 -999999 -> 2  & ID 제거)
cust_df["var3"] = cust_df["var3"].replace(-999999,2)
cust_df.drop("ID", axis=1, inplace=True)

In [8]:
# 일반 데이터와 레이블 분리
X_features = cust_df.iloc[:,:-1]
y_labels = cust_df.iloc[:,-1]

In [9]:
# 4-2. 전처리(분산 0인 컬럼 제거)
# 1. 각 컬럼의 분산 계산
stds = X_features.std()

# 2. 분산이 0인(즉, 표준편차가 0이거나 값이 모두 똑같은) 컬럼 이름 추출
zero_var_cols = stds[stds == 0].index.tolist()

print(f"분산이 0인 컬럼 개수: {len(zero_var_cols)}")
print(f"삭제할 컬럼들: {zero_var_cols}")

# 3. 해당 컬럼들 제거
X_features_clean = X_features.drop(columns=zero_var_cols)

print(f"정제 전 피처 shape: {X_features.shape}")
print(f"정제 후 피처 shape: {X_features_clean.shape}")

분산이 0인 컬럼 개수: 34
삭제할 컬럼들: ['ind_var2_0', 'ind_var2', 'ind_var27_0', 'ind_var28_0', 'ind_var28', 'ind_var27', 'ind_var41', 'ind_var46_0', 'ind_var46', 'num_var27_0', 'num_var28_0', 'num_var28', 'num_var27', 'num_var41', 'num_var46_0', 'num_var46', 'saldo_var28', 'saldo_var27', 'saldo_var41', 'saldo_var46', 'imp_amort_var18_hace3', 'imp_amort_var34_hace3', 'imp_reemb_var13_hace3', 'imp_reemb_var33_hace3', 'imp_trasp_var17_out_hace3', 'imp_trasp_var33_out_hace3', 'num_var2_0_ult1', 'num_var2_ult1', 'num_reemb_var13_hace3', 'num_reemb_var33_hace3', 'num_trasp_var17_out_hace3', 'num_trasp_var33_out_hace3', 'saldo_var2_ult1', 'saldo_medio_var13_medio_hace3']
정제 전 피처 shape: (76020, 369)
정제 후 피처 shape: (76020, 335)


In [10]:
# 값이 완전히 동일한 컬럼 중 뒤에 나온 것들의 이름을 반환
import numpy as np

def find_duplicate_columns(df):

    groups = {}
    for col in df.columns:
        v = df[col].values
        # 지문(sum/min/max)으로 후보를 먼저 좁힘
        key = (v.sum(), v.min(), v.max())
        groups.setdefault(key, []).append(col)

    dup = set()
    for cols in groups.values():
        if len(cols) < 2:
            continue
        for i in range(len(cols)):
            if cols[i] in dup:
                continue
            for j in range(i + 1, len(cols)):
                if cols[j] in dup:
                    continue
                if np.array_equal(df[cols[i]].values, df[cols[j]].values):
                    dup.add(cols[j])
    return sorted(dup)


dup_cols = find_duplicate_columns(X_features_clean)
print(f"중복 컬럼 개수: {len(dup_cols)}")
print(f"삭제할 컬럼들: {dup_cols}")

X_features_clean = X_features_clean.drop(columns=dup_cols)
print(f"정제 후 피처 shape: {X_features_clean.shape}")

중복 컬럼 개수: 29
삭제할 컬럼들: ['delta_num_reemb_var13_1y3', 'delta_num_reemb_var17_1y3', 'delta_num_reemb_var33_1y3', 'delta_num_trasp_var17_in_1y3', 'delta_num_trasp_var17_out_1y3', 'delta_num_trasp_var33_in_1y3', 'delta_num_trasp_var33_out_1y3', 'ind_var13_medio', 'ind_var18', 'ind_var25', 'ind_var26', 'ind_var29', 'ind_var29_0', 'ind_var32', 'ind_var34', 'ind_var37', 'ind_var39', 'num_var13_medio', 'num_var18', 'num_var25', 'num_var26', 'num_var29', 'num_var29_0', 'num_var32', 'num_var34', 'num_var37', 'num_var39', 'saldo_medio_var13_medio_ult1', 'saldo_var29']
정제 후 피처 shape: (76020, 306)


In [11]:
# =========================================================================
# 4. 데이터셋 단일 분할 및 대조 실험군 독립 분기 (Hold-out 공정성 확보)
# =========================================================================
from sklearn.metrics import recall_score
from xgboost import XGBClassifier


print("\n=== 데이터 분할 및 대조 실험군 독립 분기 ===")

X_train, X_test, y_train, y_test = train_test_split(
    X_features_clean, y_labels, test_size=0.2, random_state=156, stratify=y_labels
)

# [실험 A용 데이터셋 준비] 원본 버전 (var38 수치가 로그 없이 날것으로 유지됨)
X_train_norm = X_train.copy()
X_test_norm = X_test.copy()

# [실험 B용 데이터셋 준비] 로그 변환 버전 🚀
X_train_log = X_train.copy()
X_test_log = X_test.copy()

# 실험 B 세트 내부의 'var38'에만 정밀하게 로그 트랜스포메이션 주입
X_train_log['var38'] = np.log1p(X_train_log['var38'].astype(float))
X_test_log['var38'] = np.log1p(X_test_log['var38'].astype(float))

print("  - 데이터 복사 타입 충돌 및 카테고리 불일치 에러 완전 방어 완료!")


# =========================================================================
# 5. 최신 XGBoost 인프라 정의 및 대조군 학습/평가
# =========================================================================
print("\n=== XGBoost 비교 학습 및 최종 스코어 측정 ===")

def get_xgb_model():
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=100,
        eval_metric='auc',
        # 원-핫 인코딩을 완료하여 모두 수치형 칼럼으로 변환되었으므로 enable_categorical 옵션은 제외합니다.
        random_state=156
    )

# -------------------------------------------------------------------------
# 실험 A: 원본 var38 데이터셋 기반 학습 진행
# -------------------------------------------------------------------------
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
xgb_norm = get_xgb_model()
xgb_norm.fit(
    X_train_norm, y_train,
    eval_set=[(X_train_norm, y_train), (X_test_norm, y_test)],
    verbose=False
)
norm_pred_proba = xgb_norm.predict_proba(X_test_norm)[:, 1]
auc_normal = roc_auc_score(y_test, norm_pred_proba)

# 🚀 정확도 및 재현율 계산용 예측값 추출
norm_preds = xgb_norm.predict(X_test_norm)
acc_normal = accuracy_score(y_test, norm_preds)
rec_normal = recall_score(y_test, norm_preds)

# -------------------------------------------------------------------------
# 실험 B: 로그 변환 var38 데이터셋 기반 학습 진행
# -------------------------------------------------------------------------
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
xgb_log = get_xgb_model()
xgb_log.fit(
    X_train_log, y_train,
    eval_set=[(X_train_log, y_train), (X_test_log, y_test)],
    verbose=False
)
log_pred_proba = xgb_log.predict_proba(X_test_log)[:, 1]
auc_log = roc_auc_score(y_test, log_pred_proba)

# 🚀 정확도 및 재현율 계산용 예측값 추출
log_preds = xgb_log.predict(X_test_log)
acc_log = accuracy_score(y_test, log_preds)
rec_log = recall_score(y_test, log_preds)


# =========================================================================
# 6. 최종 분석 결과 종합 출력
# =========================================================================
print("\n" + "="*65)
print("             [ 최종 분석 논문 검증 결과 종합 성적표 ]")
print("="*65)
print("  구분                 |   ROC-AUC   |   정확도    |   재현율   ")
print("-"*65)
print(f"  [실험 A] 로그 전    |    {auc_normal:.4f}    |    {acc_normal:.4f}    |   {rec_normal:.4f}")
print(f"  [실험 B] 로그 후    |    {auc_log:.4f}    |    {acc_log:.4f}    |   {rec_log:.4f}")
print("-"*65)
print(f"  순수 개선 편차       |   {auc_log - auc_normal:+.4f}    |   {acc_log - acc_normal:+.4f}    |   {rec_log - rec_normal:+.4f}")
print("="*65)

# 영찬 님 결과
# 결과 : ROC AUC : 0.8517(컬럼정제전)
# 결과 : ROC AUC : 0.8531(컬럼정제후)

# 내 결과
# =================================================================
#              [ 최종 분석 논문 검증 결과 종합 성적표 ]
# =================================================================
#   구분                 |   ROC-AUC   |   정확도    |   재현율   
# -----------------------------------------------------------------
#   [실험 A] 로그 전    |    0.8520    |    0.9606    |   0.0066
#   [실험 B] 로그 후    |    0.8520    |    0.9606    |   0.0066
# -----------------------------------------------------------------
#   순수 개선 편차       |   -0.0000    |   +0.0000    |   +0.0000
# =================================================================


=== 데이터 분할 및 대조 실험군 독립 분기 ===
  - 데이터 복사 타입 충돌 및 카테고리 불일치 에러 완전 방어 완료!

=== XGBoost 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...
  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...

             [ 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8520    |    0.9606    |   0.0066
  [실험 B] 로그 후    |    0.8520    |    0.9606    |   0.0066
-----------------------------------------------------------------
  순수 개선 편차       |   -0.0000    |   +0.0000    |   +0.0000
